# EduAI Model Training
**Runtime → Change runtime type → T4 GPU** seç, sonra **Run all** bas.

In [ ]:
# 1. Unsloth quraşdır (bütün uyğunsuzluqları özü həll edir)
!pip install -q unsloth
print('✅ Unsloth hazırdır')

In [ ]:

# 3. Model yüklə və fine-tune et
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none',
)
print('✅ Model hazırdır')

def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'],tokenize=False,add_generation_prompt=False)}

train_ds = Dataset.from_list(load_jsonl('/content/data/train.jsonl')).map(fmt)
val_ds   = Dataset.from_list(load_jsonl('/content/data/val.jsonl')).map(fmt)
print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(
        output_dir='/content/eduai-finetuned',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=200,
        evaluation_strategy='steps',
        eval_steps=50,
        warmup_ratio=0.05,
        report_to='none',
    ),
)

print('Training başlayır...')
trainer.train()
model.save_pretrained('/content/eduai-finetuned')
tokenizer.save_pretrained('/content/eduai-finetuned')
print('✅ Training tamamlandı!')


In [ ]:
# 3. Model yüklə və fine-tune et
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none',
)
print('✅ Model hazırdır')

def load_jsonl(path):
    with open(path,encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'],tokenize=False,add_generation_prompt=False)}

train_ds = Dataset.from_list(load_jsonl('/content/data/train.jsonl')).map(fmt)
val_ds   = Dataset.from_list(load_jsonl('/content/data/val.jsonl')).map(fmt)
print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(
        output_dir='/content/eduai-finetuned',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=200,
        evaluation_strategy='steps',
        eval_steps=50,
        warmup_ratio=0.05,
        report_to='none',
    ),
)

print('Training başlayır...')
trainer.train()
model.save_pretrained('/content/eduai-finetuned')
tokenizer.save_pretrained('/content/eduai-finetuned')
print('✅ Training tamamlandı!')

In [ ]:
# 4. GGUF-a çevir və Drive-a yüklə
from unsloth import FastLanguageModel
from google.colab import drive

drive.mount('/content/drive')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/content/eduai-finetuned',
    max_seq_length=1024,
    load_in_4bit=True,
)

# Q4_K_M GGUF kimi saxla
model.save_pretrained_gguf(
    'eduai-model',
    tokenizer,
    quantization_method='q4_k_m',
)

import shutil, os
gguf_file = 'eduai-model/eduai-model-unsloth.Q4_K_M.gguf'
if not os.path.exists(gguf_file):
    import glob
    gguf_file = glob.glob('eduai-model/*.gguf')[0]

shutil.copy(gguf_file, '/content/drive/MyDrive/eduai-model-q4.gguf')

size = os.path.getsize('/content/drive/MyDrive/eduai-model-q4.gguf')/(1024**3)
print(f'✅ eduai-model-q4.gguf Google Drive-a yükləndi!')
print(f'📦 Ölçü: {size:.2f} GB')
print('📁 Yer: Google Drive → MyDrive → eduai-model-q4.gguf')